# Ground Truth Generation

This notebook generates synthetic user questions per chunk and stores them as JSONL ground truth for retrieval evaluation.


In [1]:
from pathlib import Path
import json
import os
import random

import chromadb
from dotenv import load_dotenv
from openai import OpenAI


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    return start


PROJECT_ROOT = find_project_root(Path.cwd())
CHROMA_PATH = PROJECT_ROOT / "backend" / "data" / "chroma_db"
GROUND_TRUTH_PATH = PROJECT_ROOT / "backend" / "data" / "eval" / "ground_truth_chunk_questions.jsonl"
BASE_COLLECTION_NAME = "estate_documents"

QUESTION_GEN_MODEL = "gpt-4o-mini"
QUESTIONS_PER_CHUNK = 5
MAX_CHUNKS = None
RANDOM_SEED = 42

load_dotenv(PROJECT_ROOT / ".env", override=True)
if not (os.getenv("OPENAI_API_KEY") or "").strip():
    raise ValueError("OPENAI_API_KEY is missing. Add it to /workspace/.env or your environment.")

client = chromadb.PersistentClient(path=str(CHROMA_PATH))
openai_client = OpenAI()

print(f"Project root: {PROJECT_ROOT}")
print(f"Chroma path: {CHROMA_PATH}")
print(f"Output file: {GROUND_TRUTH_PATH}")
print(f"Question model: {QUESTION_GEN_MODEL}")


Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given


Project root: /workspace
Chroma path: /workspace/backend/data/chroma_db
Output file: /workspace/backend/data/eval/ground_truth_chunk_questions.jsonl
Question model: gpt-4o-mini


In [2]:
def fetch_all_chunks(chroma_collection, batch_size: int = 200) -> list[dict]:
    total = chroma_collection.count()
    rows: list[dict] = []

    for offset in range(0, total, batch_size):
        batch = chroma_collection.get(
            include=["documents", "metadatas"],
            limit=batch_size,
            offset=offset,
        )
        ids = batch.get("ids", [])
        docs = batch.get("documents", [])
        metas = batch.get("metadatas", [])

        for idx, chunk_id in enumerate(ids):
            rows.append(
                {
                    "chunk_id": chunk_id,
                    "text": docs[idx] or "",
                    "metadata": metas[idx] or {},
                }
            )

    return rows


def save_jsonl(rows: list[dict], path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in rows:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")


def generate_questions_for_chunk(chunk_text: str, n_questions: int) -> list[str]:
    snippet = chunk_text.strip()[:2500]
    if not snippet:
        return []

    system_prompt = "You generate realistic user retrieval questions grounded in a provided text chunk."
    user_prompt = "\n".join(
        [
            f"Generate {n_questions} factual, diverse questions answerable from this chunk only.",
            "Return strict JSON with schema: {\"questions\": [\"...\", \"...\"]}",
            "Chunk:",
            "---",
            snippet,
            "---",
        ]
    )

    for _ in range(3):
        try:
            response = openai_client.chat.completions.create(
                model=QUESTION_GEN_MODEL,
                temperature=0.2,
                response_format={"type": "json_object"},
                messages=[
                    {"role": "system", "content": system_prompt},
                    {"role": "user", "content": user_prompt},
                ],
            )
            parsed = json.loads(response.choices[0].message.content or "{}")
            questions = [
                " ".join(q.strip().split())
                for q in parsed.get("questions", [])
                if isinstance(q, str) and q.strip()
            ]
            unique_questions = list(dict.fromkeys(questions))
            if unique_questions:
                return unique_questions[:n_questions]
        except Exception:
            continue

    return []


base_collection = client.get_or_create_collection(name=BASE_COLLECTION_NAME)
chunks = fetch_all_chunks(base_collection)
print(f"Loaded {len(chunks)} chunk(s)")

random.seed(RANDOM_SEED)
if MAX_CHUNKS is not None and MAX_CHUNKS < len(chunks):
    chunks = random.sample(chunks, MAX_CHUNKS)

print(f"Generating questions for {len(chunks)} chunk(s)")


Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event CollectionGetEvent: capture() takes 1 positional argument but 3 were given


Loaded 93 chunk(s)
Generating questions for 93 chunk(s)


In [3]:
rows: list[dict] = []

for idx, chunk in enumerate(chunks, start=1):
    chunk_id = chunk["chunk_id"]
    metadata = chunk.get("metadata", {})
    questions = generate_questions_for_chunk(chunk["text"], n_questions=QUESTIONS_PER_CHUNK)

    for q_idx, question in enumerate(questions, start=1):
        rows.append(
            {
                "question_id": f"{chunk_id}-q{q_idx:02d}",
                "question": question,
                "expected_chunk_id": chunk_id,
                "expected_document_id": metadata.get("document_id"),
                "document_type": metadata.get("document_type"),
                "page_number": metadata.get("page_number"),
            }
        )

    if idx % 10 == 0 or idx == len(chunks):
        print(f"Processed {idx}/{len(chunks)} chunks | rows so far: {len(rows)}")

save_jsonl(rows, GROUND_TRUTH_PATH)
print(f"Saved {len(rows)} ground-truth rows -> {GROUND_TRUTH_PATH}")
if rows:
    print("Sample row:")
    print(rows[0])


Processed 10/93 chunks | rows so far: 50
Processed 20/93 chunks | rows so far: 100
Processed 30/93 chunks | rows so far: 150
Processed 40/93 chunks | rows so far: 200
Processed 50/93 chunks | rows so far: 250
Processed 60/93 chunks | rows so far: 300
Processed 70/93 chunks | rows so far: 350
Processed 80/93 chunks | rows so far: 400
Processed 90/93 chunks | rows so far: 450
Processed 93/93 chunks | rows so far: 465
Saved 465 ground-truth rows -> /workspace/backend/data/eval/ground_truth_chunk_questions.jsonl
Sample row:
{'question_id': 'certificate_016-p001-c0000-q01', 'question': 'What is the document ID of the notarial certificate?', 'expected_chunk_id': 'certificate_016-p001-c0000', 'expected_document_id': 'certificate_016', 'document_type': 'family_composition_certificate', 'page_number': 1}
